In [1]:
import sympy as sp
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
%matplotlib tk

In [2]:
t = sp.Symbol('t')
g = sp.Symbol('g', positive=True)

def make_pendulum(i):
    m = sp.Symbol(f'm{i}', positive=True)
    l = sp.Symbol(f'l{i}', positive=True)
    theta = sp.Function(f'theta{i}')(t)
    return m, l, theta

In [3]:
def get_positions(thetas, lengths):
    x, y = sp.Integer(0), sp.Integer(0)
    positions = []
    
    for i, (theta, l) in enumerate(zip(thetas, lengths)):
        x = x + l * sp.sin(theta)
        y = y - l * sp.cos(theta)
        positions.append((x, y))
    
    return positions

In [4]:
def get_lagrangian(masses, lengths, n):
    thetas = []
    ls = []
    ms = []
    
    for i in range(n):
        m, l, theta = make_pendulum(i + 1)
        thetas.append(theta)
        ls.append(l)
        ms.append(m)
    
    positions = get_positions(thetas, ls)
    
    L = sp.Integer(0)
    
    for i in range(n):
        x, y = positions[i]
        
        # speed
        vx = sp.diff(x, t)
        vy = sp.diff(y, t)
        
        #kinetic energy
        T_i = sp.Rational(1, 2) * ms[i] * (vx**2 + vy**2)
        
        # potencial energy
        V_i = ms[i] * g * y
        
       
         #Lagrangian sum
        L = L + T_i - V_i
    
    return L, thetas, ms, ls

In [5]:
def get_equations_of_motion(L, thetas):
    eqs = []
    
    for theta in thetas:
        theta_dot = sp.diff(theta, t)
        
        # Euler-Lagrange equals zero
        dL_dtheta = sp.diff(L, theta)
        dL_dtheta_dot = sp.diff(L, theta_dot)
        d_dt_dL_dtheta_dot = sp.diff(dL_dtheta_dot, t)
        
        eq = d_dt_dL_dtheta_dot - dL_dtheta
        eqs.append(eq)
    
    return eqs

In [6]:
def solve_for_accelerations(eqs, thetas):
    theta_dots = [sp.diff(theta, t) for theta in thetas]
    theta_ddots = [sp.diff(sp.diff(theta, t), t) for theta in thetas]
    
   #like Gaussian elimination
    solution = sp.solve(eqs, theta_ddots)
    
    return solution, theta_ddots

In [7]:
def build_numerical_system(solution, thetas, masses, lengths, g_val=9.81, n=2):
    theta_dots = [sp.diff(theta, t) for theta in thetas]
    theta_ddots = [sp.diff(sp.diff(theta, t), t) for theta in thetas]
    
    # replace with real values
    subs = {g: g_val}
    for i in range(n):
        subs[masses[i]] = masses[i]
        subs[lengths[i]] = lengths[i]
    
    # all variables
    all_vars = thetas + theta_dots
    
    # fast numeric func
    acc_funcs = []
    for ddot in theta_ddots:
        expr = solution[ddot].subs(subs)
        expr = sp.simplify(expr)
        f = sp.lambdify(all_vars, expr, modules='numpy')
        acc_funcs.append(f)
    
    return acc_funcs

In [8]:
def simulate(acc_funcs, initial_angles, initial_velocities, t_span=(0, 20), n=2):
    # initiate the start
    y0 = list(initial_angles) + list(initial_velocities)
    
    def system(t_val, y):
        angles = y[:n]
        velocities = y[n:]
        
        # calc thе velocities in certain
        accelerations = [f(*angles, *velocities) for f in acc_funcs]
        
        # return the velocities and the accelerations
        return list(velocities) + list(accelerations)
    
    result = solve_ivp(
        system,
        t_span,
        y0,
        method='RK45',  # Runge-Kutta method
        max_step=0.01,
        rtol=1e-8, # approximate accuracy
        atol=1e-8  # approximate accuracy
    )
    
    return result

In [9]:
def run_simulation(n, mass_values, length_values, angle_values, velocity_values=None, t_span=(0, 20)):
    if velocity_values is None:
        velocity_values = [0.0] * n
    
    L, thetas, masses, lengths = get_lagrangian(None, None, n)
    eqs = get_equations_of_motion(L, thetas)
    solution, theta_ddots = solve_for_accelerations(eqs, thetas)
    
    # replace with real values
    subs = {g: 9.81}
    for i in range(n):
        subs[masses[i]] = mass_values[i]
        subs[lengths[i]] = length_values[i]
    
    solution_numeric = {k: v.subs(subs) for k, v in solution.items()}
    
    theta_dots = [sp.diff(theta, t) for theta in thetas]
    all_vars = thetas + theta_dots
    
    acc_funcs = []
    for ddot in theta_ddots:
        expr = sp.simplify(solution_numeric[ddot])
        f = sp.lambdify(all_vars, expr, modules='numpy')
        acc_funcs.append(f)
    
    result = simulate(acc_funcs, angle_values, velocity_values, t_span, n)
    return result, length_values

In [10]:
def animate_pendulum(result, length_values, n, title="N-Pendulum"):
    t_vals = result.t
    angles = result.y[:n]
    
    # calculate the x and y for every frame
    xs = np.zeros((n + 1, len(t_vals)))
    ys = np.zeros((n + 1, len(t_vals)))
    
    for frame in range(len(t_vals)):
        x, y = 0.0, 0.0
        for i in range(n):
            x = x + length_values[i] * np.sin(angles[i, frame])
            y = y - length_values[i] * np.cos(angles[i, frame])
            xs[i + 1, frame] = x
            ys[i + 1, frame] = y
    
    fig, ax = plt.subplots(figsize=(6, 6))
    total_length = sum(length_values)
    ax.set_xlim(-total_length * 1.2, total_length * 1.2)
    ax.set_ylim(-total_length * 1.2, total_length * 1.2)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.set_title(title)
    
    line, = ax.plot([], [], 'o-', lw=2, markersize=8, color='royalblue')
    trace, = ax.plot([], [], '-', lw=0.5, alpha=0.4, color='tomato')
    trace_x, trace_y = [], []
    
    def update(frame):
        line.set_data(xs[:, frame], ys[:, frame])
        trace_x.append(xs[-1, frame])
        trace_y.append(ys[-1, frame])
        trace.set_data(trace_x[-300:], trace_y[-300:])
        return line, trace
    
    ani = FuncAnimation(fig, update, frames=range(0, len(t_vals), 3),
                        interval=20, blit=True)
    plt.tight_layout()
    plt.show()
    return ani

In [11]:
result_2, lengths_2 = run_simulation(
    n=2,
    mass_values=[1.0, 1.0],
    length_values=[1.0, 1.0],
    angle_values=[np.pi/2, np.pi/4]  # 90 и 45 градуса
)

ani = animate_pendulum(result_2, lengths_2, n=2, title="Двойно махало")